# Notebook 06 — Full Evaluation & Interpretability
**Run after:** `05_train.ipynb` (needs a saved checkpoint)

This notebook runs the complete evaluation pipeline:

| Section | What it proves |
|---|---|
| 6A Standard metrics | Did the model learn to classify? |
| 6B Calibration curve | Can we trust the confidence numbers? |
| 6C Uncertainty analysis | Does high uncertainty = genuinely harder? |
| 6D Grad-CAM | Does the model look at the lesion, not artifacts? |


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')
import warnings; warnings.filterwarnings('ignore')
import torch
import torch.nn.functional as F
import numpy as np
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from collections import Counter

from data.dataset    import (load_metadata, split_dataset, get_dataloaders,
                              HAM10000Dataset, get_val_transforms, IDX_TO_CLASS)
from models.architecture import RobustMedicalClassifier
from models.losses       import CombinedLoss
from models.trainer      import get_device, validate, compute_metrics
from utils.evaluation    import (plot_calibration_curve, plot_uncertainty_analysis,
                                  plot_confusion_matrix, visualize_gradcam)

METADATA_CSV = 'dataset/HAM10000_metadata.csv'
IMAGES_DIR   = 'dataset/images'
OUTPUTS_DIR  = 'outputs'
CHECKPOINT   = 'outputs/best_model.pth'

device = get_device()


  Device: Apple M2 GPU (MPS) ✅ — training will use GPU cores


In [2]:
if not os.path.exists(CHECKPOINT):
    raise FileNotFoundError(f"No checkpoint at {CHECKPOINT}. Run notebook 05 first.")

df = load_metadata(METADATA_CSV, IMAGES_DIR)
df_train, df_val, df_test = split_dataset(df)
_, _, test_loader = get_dataloaders(df_train, df_val, df_test, batch_size=32)
class_counts = dict(Counter(df_train['label'].values))

model = RobustMedicalClassifier(num_classes=7, freeze_backbone=False)
model = model.to(device)
ckpt  = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded checkpoint from epoch {ckpt['epoch']} (val F1: {ckpt['val_f1']:.4f}) ✅")


STEP 1B: Loading HAM10000 Metadata
  Total records in CSV: 10015
  Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization', 'dataset']
  Unique lesions (lesion_id): 7470
  Unique images (image_id):   10015
  All 10015 images found on disk. ✅

  Class distribution:
    akiec  (Actinic Keratosis        ):   327 (3.3%)
    bcc    (Basal Cell Carcinoma     ):   514 (5.1%)
    bkl    (Benign Keratosis         ):  1099 (11.0%)
    df     (Dermatofibroma           ):   115 (1.1%)
    mel    (Melanoma                 ):  1113 (11.1%)
    nv     (Melanocytic Nevus        ):  6705 (66.9%)
    vasc   (Vascular Lesion          ):   142 (1.4%)

STEP 1C: Group-Based Train/Val/Test Split
WHY: Splitting by lesion_id prevents same lesion in train+test
  Train:  6959 images | 5228 unique lesions
  Val:    1529 images | 1121 unique lesions
  Test:   1527 images | 1121 unique lesions
  No data leakage detected ✅

STEP 1G: Creating DataLoaders
  Batch size: 32 (safe for M2 8GB)

  

## 6A — Standard Metrics

In [3]:
import pandas as pd

model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        output = model(images)
        probs  = F.softmax(output['logits'], dim=1)
        all_preds.extend(probs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

metrics = compute_metrics(all_preds.tolist(), all_labels.tolist(), all_probs.tolist())

print(f"F1 Score (macro): {metrics['f1_macro']:.4f}")
print(f"AUROC:            {metrics['auroc']:.4f}")

class_codes = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
rows = [{'Class': cls, 'F1 Score': round(f1, 4)}
        for cls, f1 in zip(class_codes, metrics['f1_per_class'])]
pd.DataFrame(rows).set_index('Class').style.background_gradient(
    subset=['F1 Score'], cmap='RdYlGn', vmin=0, vmax=1)


F1 Score (macro): 0.5693
AUROC:            0.9395


,F1 Score
Class,
nv,0.743900
mel,0.443000
bkl,0.539000
bcc,0.666700
akiec,0.451000
vasc,0.697000
df,0.444400


## 6B — Confusion Matrix

In [4]:
plot_confusion_matrix(all_preds, all_labels, OUTPUTS_DIR)

img = mpimg.imread(f'{OUTPUTS_DIR}/confusion_matrix.png')
plt.figure(figsize=(10, 8)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

print("Rows = True class | Columns = Predicted class")
print("Most concerning: mel predicted as nv (missed melanoma = false negative for cancer)")



  Plotting confusion matrix...
Rows = True class | Columns = Predicted class
Most concerning: mel predicted as nv (missed melanoma = false negative for cancer)


## 6C — Calibration Curve

In [6]:
mean_ece = plot_calibration_curve(all_probs, all_labels,7, OUTPUTS_DIR)

img = mpimg.imread(f'{OUTPUTS_DIR}/calibration_curves.png')
plt.figure(figsize=(14, 5)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

print(f"Mean ECE: {mean_ece:.4f}")
print(f"Status: {'Well-calibrated ✅' if mean_ece < 0.1 else 'Overconfident ⚠️ — consider temperature scaling'}")



  Computing calibration curves...
    Mean ECE: 0.2212 (overconfident ⚠️)
Mean ECE: 0.2212
Status: Overconfident ⚠️ — consider temperature scaling


## 6D — Uncertainty Analysis
If uncertainty is meaningful: wrong predictions should have **higher** uncertainty than correct ones.
The accuracy-vs-uncertainty curve should **decline** from left to right.


In [8]:
# Number of MC passes — use 15 for quality, 5 for speed
N_PASSES = 5   # increase to 15 or 20 for final results

uncertainties, correct_mask = [], []
model.train()   # keep MC Dropout active

with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        result = model.predict_with_uncertainty(images, n_passes=N_PASSES)
        uncertainties.extend(result['mc_uncertainty'].cpu().numpy())
        correct = (result['predicted_class'].cpu().numpy() == labels.numpy())
        correct_mask.extend(correct.astype(int))

uncertainties = np.array(uncertainties)
correct_mask  = np.array(correct_mask)

m_correct = uncertainties[correct_mask==1].mean()
m_wrong   = uncertainties[correct_mask==0].mean()
print(f"Mean uncertainty — Correct: {m_correct:.4f} | Wrong: {m_wrong:.4f}")
print(f"{'✅ Wrong predictions have higher uncertainty — model is self-aware!' if m_wrong > m_correct else '⚠️ Similar uncertainty — may need more training'}")


Python(96981) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(96982) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Mean uncertainty — Correct: 0.0023 | Wrong: 0.0028
✅ Wrong predictions have higher uncertainty — model is self-aware!


In [9]:
plot_uncertainty_analysis(uncertainties, correct_mask, OUTPUTS_DIR)

img = mpimg.imread(f'{OUTPUTS_DIR}/uncertainty_analysis.png')
plt.figure(figsize=(13, 5)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()



  Plotting uncertainty analysis...
    Mean uncertainty — Correct: 0.0023 | Wrong: 0.0028
    Wrong predictions have higher uncertainty ✅ — model is self-aware


## 6E — Grad-CAM Visualization
Shows where the model looked when making each prediction.
**Red = high attention. Green = low attention.**

What we want to see:
- ✅ Red concentrated on the dark irregular lesion
- ❌ Red on hair, rulers, or background = shortcut learning


In [10]:
test_dataset = HAM10000Dataset(df_test, transform=get_val_transforms())
visualize_gradcam(model, test_dataset, device, OUTPUTS_DIR, n_per_class=3)

img = mpimg.imread(f'{OUTPUTS_DIR}/gradcam_per_class.png')
plt.figure(figsize=(14, 24)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()
print("\n→ NEXT: Open notebook 07_ablation.ipynb")



  Generating Grad-CAM visualizations...
    Grad-CAM saved to: outputs/gradcam_per_class.png

→ NEXT: Open notebook 07_ablation.ipynb
